# Hybrid Fusion: Combining Transformer and Lexicon-Based Scores

This notebook combines the DistilBERT baseline outputs with snippet/lexicon scores
to create a hybrid sentiment representation. The goal is to balance transformer confidence
with rule-based interpretability.

In [1]:
# 1. Setup
import pandas as pd

# Load transformer baseline predictions (5-class)
baseline = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# Load snippet/lexicon scores (5-class)
snippet = pd.read_csv("../data/processed/snippet_scores_5class.csv")

print(f"✅ Loaded Baseline predictions: {baseline.shape}")
print(f"✅ Loaded Snippet scores: {snippet.shape}")

✅ Loaded Baseline predictions: (14166, 6)
✅ Loaded Snippet scores: (14166, 7)


In [2]:
# 2. Quick sanity checks
print("Baseline columns:", baseline.columns.tolist())
print("Snippet columns:", snippet.columns.tolist())

print("Companies:", snippet['company'].unique())
print("Years:", snippet['year'].unique())

Baseline columns: ['company', 'year', 'sentence', 'label', 'score', 'sentiment_5class']
Snippet columns: ['company', 'year', 'sentence', 'label', 'score', 'sentiment_5class', 'lexicon_5class']
Companies: ['Google' 'HSBC' 'Nestle']
Years: [2022 2023 2024]


In [29]:
bert_doc = (
    baseline.groupby(["company", "year"])
    .agg({
        "score": "mean"  # BERT confidence score
    })
    .reset_index()
    .rename(columns={"score": "bert_mean_score"})
)


from collections import defaultdict


lexicon_scores = []
for _, row in baseline.iterrows(): 
    company, year, sentence = row['company'], row['year'], row['sentence']
    lex_score = calc_lex_score(sentence)  
    lexicon_scores.append({
        'company': company,
        'year': year, 
        'lex_score': lex_score
    })


lexicon_df = pd.DataFrame(lexicon_scores)
snippet_doc = (
    lexicon_df.groupby(["company", "year"])
    .agg({
        "lex_score": "mean"
    })
    .reset_index()
    .rename(columns={"lex_score": "lexicon_score"})
)

In [33]:
# 4. Merge Baseline (doc-level) + Lexicon (doc-level)
hybrid_df = pd.merge(snippet_doc, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid doc-level DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())


✅ Hybrid doc-level DataFrame created: (9, 4)
  company  year  lexicon_score  bert_mean_score
0  Google  2022       1.358885         0.898745
1  Google  2023       1.248726         0.937662
2  Google  2024      -0.259122         0.927499
3    HSBC  2022       1.385965         0.942709
4    HSBC  2023       0.773054         0.938589


In [34]:
# 4. Merge Baseline (doc-level) + Lexicon (doc-level)
hybrid_df = pd.merge(snippet_doc, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid doc-level DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())

# --- Normalize lexicon scores (min-max scaling) ---
min_val = hybrid_df["lexicon_score"].min()
max_val = hybrid_df["lexicon_score"].max()

hybrid_df["lexicon_score_normalized"] = hybrid_df["lexicon_score"].apply(
    lambda x: (x - min_val) / (max_val - min_val) if max_val != min_val else 0.5
)

# --- Fusion (0.7 BERT + 0.3 Lexicon) ---
hybrid_df["hybrid_score"] = (
    0.7 * hybrid_df["bert_mean_score"] + 0.3 * hybrid_df["lexicon_score_normalized"]
)

# --- 5-class mapping (0–1 scale) ---
def hybrid_to_5class(score, high=0.8, low=0.2):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= 0.6:
        return "POSITIVE"
    elif score >= 0.4:
        return "NEUTRAL"
    elif score >= low:
        return "NEGATIVE"
    else:
        return "VERY NEGATIVE"

hybrid_df["hybrid_5class"] = hybrid_df["hybrid_score"].apply(hybrid_to_5class)

# --- Save final doc-level hybrid scores ---
out_path = "../data/processed/hybrid_scores_5class.csv"
hybrid_df.to_csv(out_path, index=False)

print(f"✅ Saved hybrid 5-class scores to {out_path}")
print(hybrid_df[["company","year","lexicon_score","lexicon_score_normalized","bert_mean_score","hybrid_score","hybrid_5class"]])

✅ Hybrid doc-level DataFrame created: (9, 4)
  company  year  lexicon_score  bert_mean_score
0  Google  2022       1.358885         0.898745
1  Google  2023       1.248726         0.937662
2  Google  2024      -0.259122         0.927499
3    HSBC  2022       1.385965         0.942709
4    HSBC  2023       0.773054         0.938589
✅ Saved hybrid 5-class scores to ../data/processed/hybrid_scores_5class.csv
  company  year  lexicon_score  lexicon_score_normalized  bert_mean_score  \
0  Google  2022       1.358885                  0.869428         0.898745   
1  Google  2023       1.248726                  0.810235         0.937662   
2  Google  2024      -0.259122                  0.000000         0.927499   
3    HSBC  2022       1.385965                  0.883979         0.942709   
4    HSBC  2023       0.773054                  0.554634         0.938589   
5    HSBC  2024      -0.206948                  0.028036         0.939063   
6  Nestle  2022       1.544554                  0.96

In [24]:
# 6 Improved Sentence-level Lexicon with Negations and Intensifiers
#In this step, we expand the lexicon method with negations and intensifiers.

import pandas as pd

# Load baseline sentence-level predictions
baseline = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# --- ESG Lexicon ---
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

# --- Lexicon scoring function ---
def calc_lex_score(sentence):
    tokens = str(sentence).lower().split()
    score = 0
    i = 0
    while i < len(tokens):
        token = tokens[i]

        # Intensifier check
        if token in INTENSIFIERS and i+1 < len(tokens):
            next_token = tokens[i+1]
            if next_token in ESG_LEXICON["positive"]:
                score += 20
                i += 1
            elif next_token in ESG_LEXICON["negative"]:
                score -= 20
                i += 1

        # Negation check
        elif token in NEGATIONS and i+1 < len(tokens):
            next_token = tokens[i+1]
            if next_token in ESG_LEXICON["positive"]:
                score -= 10
                i += 1
            elif next_token in ESG_LEXICON["negative"]:
                score += 10
                i += 1

        # Normal lexicon check
        elif token in ESG_LEXICON["positive"]:
            score += 10
        elif token in ESG_LEXICON["negative"]:
            score -= 10

        i += 1
    return score

# Add lexicon score column
baseline["lex_score"] = baseline["sentence"].apply(calc_lex_score)

# --- Override hybrid fusion ---
def hybrid_override(row):
    bert_scaled = row["score"] * 100.0
    lex_score = row["lex_score"]
    
    if lex_score != 0:
        return 0.5 * bert_scaled + 0.5 * lex_score
    else:
        return bert_scaled

baseline["hybrid_score"] = baseline.apply(hybrid_override, axis=1)

# --- Map scores to 5-class ---
def score_to_5class(score, high=70, low=20):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= low:
        return "POSITIVE"
    elif score <= -high:
        return "VERY NEGATIVE"
    elif score <= -low:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

baseline["lexicon_5class"] = baseline["lex_score"].apply(score_to_5class)
baseline["hybrid_5class"]  = baseline["hybrid_score"].apply(score_to_5class)

# Save updated files
baseline.to_csv("../data/processed/lexicon_5class_sentencelevel.csv", index=False)
baseline.to_csv("../data/processed/hybrid_5class_sentencelevel.csv", index=False)

print("💾 Saved: lexicon_5class_sentencelevel.csv & hybrid_5class_sentencelevel.csv (override weighting)")

💾 Saved: lexicon_5class_sentencelevel.csv & hybrid_5class_sentencelevel.csv (override weighting)
